In [ ]:
from pathlib import Path 
import numpy as np 

SPLITS =("train","val","test")
DATASETS =("BreastMNIST","BUSUCLM","INbreast")

BRANCHES ={
"classical_resnet":("classical","resnet"),
"classical_shallow":("classical","shallow"),
"quantum_basic":("quantum","basic"),
"quantum_trainable":("quantum","trainable"),
}

COMBINATIONS =[
("quantum_basic","classical_shallow","basic_shallow"),
("quantum_basic","classical_resnet","basic_resnet"),
("quantum_trainable","classical_shallow","trainable_shallow"),
("quantum_trainable","classical_resnet","trainable_resnet"),
]

OFFLINE_ROOT =Path ("/home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/models/fusion/offline")
print (f"Offline root: {OFFLINE_ROOT }")
print (f"\nCombinations to process:")
for q ,c ,name in COMBINATIONS :
    print (f"  - {name}: {q} + {c}")

In [ ]:

LABELS_ROOT =Path ("/home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/labels")
reference_labels ={}

print (f"Loading reference labels from: {LABELS_ROOT }\n")

for dataset_name in DATASETS :
    reference_labels [dataset_name ]={}
    for split in SPLITS :
        y_path =LABELS_ROOT /dataset_name /f"{split }_labels.npy"
        if y_path .exists ():
            y =np .load (y_path )
            reference_labels [dataset_name ][split ]=y 
            print (f"{dataset_name } | {split :5s} - shape: {y .shape } - unique: {np .unique (y )}")
        else :
            print (f"{dataset_name } | {split :5s} - FILE NOT FOUND: {y_path }")

print (f"\nLabels loaded successfully!")
print (f"Access: reference_labels[dataset][split]")

In [ ]:
def load_branch_features (dataset :str ,branch_key :str ,split :str ):
    family ,model =BRANCHES [branch_key ]
    folder =OFFLINE_ROOT /dataset /family /model 

    x_path =folder /f"{split }_features.npy"

    if not x_path .exists ():
        raise FileNotFoundError (f"Feature file not found: {x_path }")

    x =np .load (x_path )
    return x 


def concat_pair_and_save (dataset :str ,quantum_key :str ,classical_key :str ,combination_name :str ):
    output_dir =OFFLINE_ROOT /dataset /"concatenated"/combination_name 
    output_dir .mkdir (parents =True ,exist_ok =True )

    print (f"  {combination_name }:")

    for split in SPLITS :

        x_quantum =load_branch_features (dataset ,quantum_key ,split )
        x_classical =load_branch_features (dataset ,classical_key ,split )


        x_concat =np .concatenate ([x_quantum ,x_classical ],axis =1 )


        y_ref =reference_labels [dataset ][split ]


        if x_concat .shape [0 ]!=y_ref .shape [0 ]:
            raise ValueError (
            f"Inconsistency between concatenated features and labels in {dataset } | {split }: "
            f"features={x_concat .shape [0 ]} labels={y_ref .shape [0 ]}"
            )

        x_out =output_dir /f"{split }_features_concat.npy"
        y_out =output_dir /f"{split }_labels.npy"

        np .save (x_out ,x_concat )
        np .save (y_out ,y_ref )

        print (
        f"    [{split }] concat shape: {x_concat .shape } | "
        f"quantum: {x_quantum .shape [1 ]} + classical: {x_classical .shape [1 ]} = {x_concat .shape [1 ]}"
        )


def process_all_combinations_for_dataset (dataset :str ):
    print (f"\n{'='*60 }")
    print (f"Dataset: {dataset }")
    print (f"{'='*60 }")

    for quantum_key ,classical_key ,combination_name in COMBINATIONS :
        concat_pair_and_save (dataset ,quantum_key ,classical_key ,combination_name )

    print (f"\n✓ {dataset } completed!")

In [ ]:
for dataset_name in DATASETS :
    process_all_combinations_for_dataset (dataset_name )

print ("\n"+"="*60 )
print ("✓ DONE: Offline features concatenated pairwise!")
print ("="*60 )
print ("\nCreated folders:")
for dataset_name in DATASETS :
    for _ ,_ ,comb_name in COMBINATIONS :
        path =OFFLINE_ROOT /dataset_name /"concatenated"/comb_name 
        print (f"  {dataset_name }/{comb_name }/")